1. Charger et prétraiter l'ensemble de données MNIST
Importation des bibliothèques

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

from sklearn.metrics import confusion_matrix

In [ ]:
# Chargement du dataset MNIST
(X_train, y_train), (X_test, y_test) = mnist.load_data()

print("Train :", X_train.shape)
print("Test :", X_test.shape)

In [ ]:
# Affichage de quelques images
plt.figure(figsize=(10,5))

for i in range(10):
    plt.subplot(2,5,i+1)
    plt.imshow(X_train[i], cmap='gray')
    plt.title(f"Label : {y_train[i]}")
    plt.axis('off')

plt.show()

In [ ]:
# Normalisation des pixels
X_train = X_train / 255.0
X_test = X_test / 255.0

In [ ]:
print(X_train.min())
print(X_train.max())

In [ ]:
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

2. Construire un réseau neuronal entièrement connecté
Importation de Keras

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten

In [ ]:
# Création du modèle
model = Sequential([
    
    Flatten(input_shape=(28,28)),
    
    Dense(
        128,
        activation='relu'
    ),
    
    Dense(
        64,
        activation='relu'
    ),
    
    Dense(
        10,
        activation='softmax'
    )
])

In [ ]:
# Résumé du modèle
model.summary()

In [ ]:
# Compilation
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# 3. Entraîner le réseau neuronal
history = model.fit(
    X_train,
    y_train_cat,
    epochs=10,
    validation_split=0.2,
    batch_size=32
)

In [ ]:
# Évolution de la perte
plt.figure(figsize=(8,5))

plt.plot(
    history.history['loss'],
    label='Train Loss'
)

plt.plot(
    history.history['val_loss'],
    label='Validation Loss'
)

plt.title("Loss")
plt.legend()

plt.show()

In [ ]:
# Évolution de la précision
plt.figure(figsize=(8,5))

plt.plot(
    history.history['accuracy'],
    label='Train Accuracy'
)

plt.plot(
    history.history['val_accuracy'],
    label='Validation Accuracy'
)

plt.title("Accuracy")
plt.legend()

plt.show()

4. Évaluer les performances du modèle
Précision sur les données de test

In [ ]:
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test_cat
)

print("Test Accuracy :", test_accuracy)

In [ ]:
# Prédictions
y_pred_proba = model.predict(X_test)

y_pred = np.argmax(
    y_pred_proba,
    axis=1
)

In [ ]:
# Matrice de confusion
cm = confusion_matrix(
    y_test,
    y_pred
)

plt.figure(figsize=(10,8))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.xlabel("Prédiction")
plt.ylabel("Réel")
plt.title("Matrice de confusion")

plt.show()

In [ ]:
# Visualisation des erreurs
errors = np.where(
    y_pred != y_test
)[0]

print(
    "Nombre d'erreurs :",
    len(errors)
)

In [ ]:
# Afficher quelques chiffres mal classés
plt.figure(figsize=(12,8))

for i in range(9):
    
    idx = errors[i]

    plt.subplot(3,3,i+1)

    plt.imshow(
        X_test[idx],
        cmap='gray'
    )

    plt.title(
        f"Vrai:{y_test[idx]} / Pred:{y_pred[idx]}"
    )

    plt.axis('off')

plt.show()

In [ ]:
# Analyse des chiffres les plus difficiles
misclassified = {}

for true, pred in zip(y_test, y_pred):
    
    if true != pred:
        
        misclassified[true] = (
            misclassified.get(true,0) + 1
        )

print(misclassified)